In [ ]:
from databallpy import get_game

game = get_game(
    tracking_data_loc="../data/tracab_tracking.dat",
    tracking_metadata_loc="../data/tracab_metadata.xml",
    tracking_data_provider="tracab",
    event_data_loc="../data/opta_f24.xml",
    event_metadata_loc="../data/opta_f7.xml",
    event_data_provider="opta",
)

In [ ]:
print(game)
print(f"Game name: {game.name}\n")

# Synchronize tracking and event data with DataBallPy defaults


In [ ]:
# Use DataBallPy's default preprocessing and synchronization parameters.
game.synchronise_tracking_and_event_data()

print(game.event_data.head())

game.event_data.to_csv("../data/event_data.csv", index=False)
game.tracking_data.to_csv("../data/tracking_data.csv", index=False)

# Match metadata (meta.json)

Read by `phase.meta`: team names and event `team_id`s (home = tracking team 0, the labels' reference team),
kick-off of each period as a 0-based row of the tracking export (= frame of the fused 25 Hz CSV), pitch size,
frame rate, and the roster (player column -> goalkeeper) for `phase.prep --meta`. Used by `phase.events.load(home_team_id=...)`, `tools/adj/make_pack.py --meta` and
`tools/adj/apply_decisions.py`.

In [ ]:
import json
import xml.etree.ElementTree as ET

import numpy as np

TRACKING_META = "../data/tracab_metadata.xml"
match_el = ET.parse(TRACKING_META).getroot().find(".//match")

td = game.tracking_data
frames = td["frame"].to_numpy()  # provider frame ids; the fused CSV row = position in this export
period_start, period_end = {}, {}
for p in game.periods.itertuples():
    rows = np.flatnonzero(frames == p.start_frame)
    if len(rows):  # periods that were not played have no frame
        period_start[str(int(p.period_id))] = int(rows[0])
        end = np.flatnonzero(frames == p.end_frame)
        period_end[str(int(p.period_id))] = int(end[0]) if len(end) else len(frames) - 1

frame_rate = getattr(td, "frame_rate", None) or float(match_el.get("iFrameRateFps"))
meta = dict(
    match_id=str(match_el.get("iId")),
    home_team=dict(name=game.home_team_name, id=int(game.home_team_id)),
    away_team=dict(name=game.away_team_name, id=int(game.away_team_id)),
    period_start_frame=period_start,
    period_end_frame=period_end,
    pitch=dict(length=float(game.pitch_dimensions[0]), width=float(game.pitch_dimensions[1])),
    frame_rate=float(frame_rate),
)
# roster: fused CSV column '<side>_<side>_<shirt number>' (databallpy tracking column '<side>_<shirt>_x')
players = []
for side, df in (("home", game.home_players), ("away", game.away_players)):
    for r in df.itertuples():
        if f"{side}_{int(r.shirt_num)}_x" not in td.columns:  # did not play
            continue
        players.append(dict(column=f"{side}_{side}_{int(r.shirt_num)}", team=side, shirt_num=int(r.shirt_num),
                            name=r.full_name, position=str(r.position),
                            is_gk=str(r.position).lower().startswith("goal")))
tracked = {c[:-2] for c in td.columns if c.startswith(("home_", "away_")) and c.endswith("_x")}
assert tracked == {p["column"].split("_", 1)[1] for p in players}, "tracking columns and roster differ"
assert [p["team"] for p in players if p["is_gk"]].count("home") >= 1, "no home goalkeeper in the roster"
meta["players"] = players
with open("../data/meta.json", "w") as f:
    json.dump(meta, f, indent=1, ensure_ascii=False)
meta

# Load and parse China PR vs Japan (11/19/2024) event data

In [ ]:
import sys
sys.path.insert(0, "../tools")
from parse_events_to_df import parse_match_data

xml_file = "../比赛数据/CHINE-JAPON-191124/Match_CHINE-JAPON-191124_EVENT_2668693.xml"
df_players, df_events = parse_match_data(xml_file)

display(df_players.head())
df_events